In [1]:
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

/Users/juliasbardelatti/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df_avisos = pd.read_csv("./dados_ondas_calor.csv")
df_mortalidade = pd.read_csv("./dados_processados_SIM.csv")
df_avisos['municipio'] = df_avisos['codigo_ibge_municipio'].astype(str).str[:6]
df_mortalidade['municipio'] = df_mortalidade['CODMUNRES'].astype(str).str[:6]

/var/folders/gy/cxx2_pv95ys52g0qvnjgcsqm0000gn/T/ipykernel_15301/3563705865.py:2: DtypeWarning: Columns (30) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mortalidade = pd.read_csv("./dados_processados_SIM.csv")


In [3]:
df_avisos['data'] = pd.to_datetime(df_avisos['data_envio']).dt.tz_localize(None).dt.normalize()
df_mortalidade['data'] = pd.to_datetime(df_mortalidade['DTOBITO']).dt.tz_localize(None).dt.normalize()

In [4]:
df_avisos = df_avisos.query("data >= '2022-01-01' and data <= '2024-12-31'")
df_mortalidade = df_mortalidade.query("data >= '2022-01-01' and data <= '2024-12-31'")

In [5]:
regex_cardiovascular = r'^I[0-9][0-9]'
df_mortalidade_filtrado = df_mortalidade.copy()
df_mortalidade_filtrado = df_mortalidade_filtrado[df_mortalidade['CAUSABAS'].str.contains(regex_cardiovascular, na=False, regex=True)]

In [6]:
len(df_mortalidade_filtrado)

1154282

### Agrupando municipio e dia
- Sempre priorizando a severidade do dia conforme ordem (Extremo, Severo, Moderado)

In [7]:
severidade_prioridade = {
    'Extreme': 1,
    'Severe': 2,
    'Moderate': 3,
}

df_avisos['prioridade'] = df_avisos['severidade'].map(severidade_prioridade)

avisos_prioritario = df_avisos.loc[
    df_avisos.groupby(['municipio', 'data'])['prioridade'].idxmin()
].copy()

mortes_agrupadas = df_mortalidade_filtrado.groupby(['municipio', 'data']).size().reset_index(name='qtd_mortes')

df_full_1 = pd.merge(
    avisos_prioritario[['municipio', 'data', 'severidade']],  
    mortes_agrupadas,
    on=['municipio', 'data'],
    how='outer'
).fillna({'severidade': 'Sem aviso', 'qtd_mortes': 0})

df_full_1['teve_aviso'] = (df_full_1['severidade'] != 'Sem aviso').astype(int)
df_full_1['data'] = pd.to_datetime(df_full_1['data'])


### Para cada municipio vamos adicionar todos os dias do ano de 2022 até 2024 (mesmo sem aviso ou óbito)

In [8]:
data_inicio = df_full_1['data'].min()
data_fim = df_full_1['data'].max()
datas_completas = pd.date_range(start=data_inicio, end=data_fim, freq='D')
municipios = df_full_1['municipio'].unique()

lista_dfs = []
for mun in municipios:
    df_mun = df_full_1[df_full_1['municipio'] == mun].set_index('data')
    df_mun = df_mun.reindex(datas_completas)
    df_mun['municipio'] = mun
    df_mun['data'] = df_mun.index
    lista_dfs.append(df_mun.reset_index(drop=True))

df_completo = pd.concat(lista_dfs, ignore_index=True)

In [9]:
df_completo['severidade'] = df_completo['severidade'].fillna('Sem aviso')
df_completo['teve_aviso'] = (df_completo['severidade'] != 'Sem aviso').astype(int)
df_completo['qtd_mortes'] = df_completo['qtd_mortes'].fillna(0).astype(int)

### Criar coluna de lags 1, 2 e 3 dias

In [10]:
lags = [1, 2, 3]
cols_lag = ['teve_aviso']

df_completo = df_completo.sort_values(['municipio', 'data']).reset_index(drop=True)

for col in cols_lag:
    for lag in lags:
        df_completo[f'{col}_lag_{lag}'] = df_completo.groupby('municipio')[col].shift(lag).fillna(0).astype(int)

df_completo['aviso_lag_0_a_3'] = df_completo[ ['teve_aviso'] + [f'teve_aviso_lag_{l}' for l in lags] ].max(axis=1).astype(int)

### Adicionar dados de regiões de saúde

In [11]:
df_regioes_saude = pd.read_csv("./dados/macroregiao_de_saude.csv", sep=";")
df_regioes_saude = df_regioes_saude[['sg_uf', 'co_uf', 'cod_macrorregiao_de_saude', 'regiao_de_saude', 'macrorregiao_de_saude', 'cod_municipio']]
df_regioes_saude.columns = ['sg_uf', 'co_uf', 'cod_macrorregiao_de_saude', 'regiao_de_saude', 'macrorregiao_de_saude', 'municipio']

In [12]:
df_completo['municipio'] = df_completo['municipio'].astype(int)
df_regioes_saude['municipio'] = df_regioes_saude['municipio'].astype(int)


df_full = pd.merge(
    df_completo,
    df_regioes_saude,
    on=['municipio'],
    how='inner'
)

In [13]:
len(df_full)

6104720

### Indicadores

In [14]:
df_full['aviso_lag_0_a_3'] = df_full[ ['teve_aviso'] + [f'teve_aviso_lag_{l}' for l in lags] ].max(axis=1).astype(int)

In [15]:
media_com_aviso = round(df_full[df_full['aviso_lag_0_a_3'] == 1]['qtd_mortes'].mean(),2)
media_sem_aviso = round(df_full[df_full['aviso_lag_0_a_3'] == 0]['qtd_mortes'].mean(),2)

RR = media_com_aviso / media_sem_aviso
aumento_percentual = (RR - 1) * 100

print(f"Risco relativo (RR): {RR:.2f}")
print(f"Aumento percentual do risco em dias com aviso: {aumento_percentual:.1f}%")

Risco relativo (RR): 1.16
Aumento percentual do risco em dias com aviso: 15.8%


In [60]:
#110001
#2023-08-22	

### Avaliando os lags de 3 dias por severidade

In [16]:
df_full['sev_extreme'] = (df_full['severidade'] == 'Extreme').astype(int)
df_full['sev_severe'] = (df_full['severidade'] == 'Severe').astype(int)
df_full['sev_moderate'] = (df_full['severidade'] == 'Moderate').astype(int)
df_full['sev_sem_aviso'] = (df_full['severidade'] == 'Sem aviso').astype(int)  

In [17]:
lags = [1, 2, 3]
for sev in ['sev_extreme', 'sev_severe', 'sev_moderate', 'sev_sem_aviso']:
    for lag in lags:
        df_full[f'{sev}_lag_{lag}'] = df_full.groupby('municipio')[sev].shift(lag).fillna(0).astype(int)


In [18]:
for sev in ['sev_extreme', 'sev_severe', 'sev_moderate', 'sev_sem_aviso']:
    cols_lag = [sev] + [f'{sev}_lag_{lag}' for lag in lags]
    df_full[f'{sev}_lag_0_a_3'] = df_full[cols_lag].max(axis=1).astype(int)


In [19]:
print("Média mortes dias com qualquer aviso (0 a 3 dias):", 
      round(df_full[df_full['aviso_lag_0_a_3'] == 1]['qtd_mortes'].mean(),2))
print("Média mortes dias sem aviso (0 a 3 dias):", 
      round(df_full[df_full['aviso_lag_0_a_3'] == 0]['qtd_mortes'].mean(),2))
print("Média mortes dias com aviso extremo (0 a 3 dias):", 
      round(df_full[df_full['sev_extreme_lag_0_a_3'] == 1]['qtd_mortes'].mean(),2))
print("Média mortes dias com aviso severo (0 a 3 dias):", 
      round(df_full[df_full['sev_severe_lag_0_a_3'] == 1]['qtd_mortes'].mean(),2))
print("Média mortes dias com aviso moderado (0 a 3 dias):", 
      round(df_full[df_full['sev_moderate_lag_0_a_3'] == 1]['qtd_mortes'].mean(),2))

Média mortes dias com qualquer aviso (0 a 3 dias): 0.22
Média mortes dias sem aviso (0 a 3 dias): 0.19
Média mortes dias com aviso extremo (0 a 3 dias): 0.24
Média mortes dias com aviso severo (0 a 3 dias): 0.2
Média mortes dias com aviso moderado (0 a 3 dias): 0.22


In [65]:
def get_severidade_prioritaria(row):
    if row['sev_extreme_lag_0_a_3'] == 1:
        return 'Extreme'
    elif row['sev_severe_lag_0_a_3'] == 1:
        return 'Severe'
    elif row['sev_moderate_lag_0_a_3'] == 1:
        return 'Moderate'
    else:
        return 'Sem aviso'

df_full['severidade_lag_0_a_3'] = df_full.apply(get_severidade_prioritaria, axis=1)


In [67]:
df_full=df_full[['municipio', 'severidade', 'qtd_mortes', 'teve_aviso', 'data','aviso_lag_0_a_3', 'sg_uf', 'co_uf', 'cod_macrorregiao_de_saude','regiao_de_saude', 'macrorregiao_de_saude', 'sev_extreme_lag_0_a_3','sev_severe_lag_0_a_3', 'sev_moderate_lag_0_a_3', 'sev_sem_aviso_lag_0_a_3', 'teve_aviso_lag_3']]

### Modelos

In [69]:
formula = 'qtd_mortes ~ teve_aviso_lag_3'
import statsmodels.formula.api as smf
modelo = smf.mixedlm(formula, df_full, groups=df_full['cod_macrorregiao_de_saude'])
resultado = modelo.fit()
print(resultado.summary())


           Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: qtd_mortes   
No. Observations: 6104720 Method:             REML         
No. Groups:       120     Scale:              0.7780       
Min. group size:  1096    Log-Likelihood:     -7896966.5479
Max. group size:  161112  Converged:          Yes          
Mean group size:  50872.7                                  
-----------------------------------------------------------
                  Coef.  Std.Err.   z   P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept          0.872    0.376 2.322 0.020  0.136  1.608
teve_aviso_lag_3   0.010    0.003 2.914 0.004  0.003  0.016
Group Var         16.926    1.296                          



In [70]:
formula = 'qtd_mortes ~ sev_extreme_lag_0_a_3 + sev_severe_lag_0_a_3 + sev_moderate_lag_0_a_3'
import statsmodels.formula.api as smf
modelo = smf.mixedlm(formula, df_full, groups=df_full['cod_macrorregiao_de_saude'])
resultado = modelo.fit()
print(resultado.summary())


             Mixed Linear Model Regression Results
Model:               MixedLM  Dependent Variable:  qtd_mortes   
No. Observations:    6104720  Method:              REML         
No. Groups:          120      Scale:               0.7780       
Min. group size:     1096     Log-Likelihood:      -7896963.1696
Max. group size:     161112   Converged:           Yes          
Mean group size:     50872.7                                    
----------------------------------------------------------------
                       Coef.  Std.Err.   z   P>|z| [0.025 0.975]
----------------------------------------------------------------
Intercept               0.872    0.376 2.321 0.020  0.136  1.608
sev_extreme_lag_0_a_3   0.015    0.003 4.835 0.000  0.009  0.022
sev_severe_lag_0_a_3    0.004    0.003 1.327 0.184 -0.002  0.010
sev_moderate_lag_0_a_3  0.009    0.003 2.602 0.009  0.002  0.015
Group Var              16.926    1.296                          

